# Personal Information
Name: **Buğra Sipahioğlı**

StudentID: **14318334**

Email: [**bugra.sipahioglu@student.uva.nl**](youremail@student.uva.nl)

Submitted on: **23.03.2026**

# Data Context


# Data Description

In [ ]:
# Imports
import os
import numpy as np
import pandas as pd
from tell_me_again import StoryDataset, SimilarityDataset
import json
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.probability import FreqDist
from pathlib import Path
import nltk

# Download package(s)
nltk.download("punkt_tab", quiet=True)

## Dataset 1: MAVEN

### Data Loading

In [ ]:
# Define the core directories
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("."), ".."))
MAVEN_DIR_RAW = os.path.join(PROJECT_ROOT, "data", "raw", "MAVEN")

# Load the data
df_train = pd.read_json(os.path.join(MAVEN_DIR_RAW, "train.jsonl"), lines=True)
df_val = pd.read_json(os.path.join(MAVEN_DIR_RAW, "valid.jsonl"), lines=True)
df_test = pd.read_json(os.path.join(MAVEN_DIR_RAW, "test.jsonl"), lines=True)

### Analysis 1: Corpus-level Overview
We inspect corpus size (number of documents and variables per split), variable types, and missing values. We also show one example document so the structure of content, events, and negative_triggers is clear.


*MAVEN is a document-level event-detection dataset: each row is one document with title, id, and content (tokenized sentences); in train/validation each document has annotated events (with type and trigger span) and negative_triggers; the test set provides only content and candidates for trigger (and possibly type) prediction.*

In [ ]:
# Analyse the shapes of train, valid, and test splits
print("\033[1;33m\nShapes of the splits:\033[0m")
for name, df in [("Train", df_train), ("Valid", df_val), ("Test", df_test)]:
    print(f"  {name}: {df.shape}")

# Print the variable types and non-null counts for all splits
print("\033[1;33m\nVariable types and non-null counts (train):\033[0m"); print(df_train.info())
print("\033[1;33m\nVariable types and non-null counts (validation):\033[0m"); print(df_val.info())
print("\033[1;33m\nVariable types and non-null counts (test):\033[0m"); print(df_test.info())

# Print the missing values per variable
print("\033[1;33m\nMissing values per variable (train):\033[0m\n" + str(df_train.isna().sum()))
print("\033[1;33m\nMissing values per variable (validation):\033[0m\n" + str(df_val.isna().sum()))
print("\033[1;33m\nMissing values per variable (test):\033[0m\n" + str(df_test.isna().sum()))

# Print one full document entity from the train split for inspection
print("\033[1;33m\nExample document from train split:\033[0m")
print(json.dumps(df_train.iloc[0].to_dict(), indent=2, ensure_ascii=False))

### Analysis 2: Pre-variable-level Overview

We describe the corpus before defining model-ready variables: document-level lengths (sentences, tokens), token-count statistics (vocabulary size, hapax count), and annotation counts (events and negative triggers per document). This characterises the raw text and annotations that will later be turned into features.


*We characterised the MAVEN corpus at a pre-variable level by computing, for each document in the train and validation splits, the number of sentences, total tokens, in-document vocabulary size, number of hapax legomena, number of events, and number of negative triggers. Summary statistics and side-by-side comparison showed that train and validation have similar distributions for these quantities, with no missing values in the analysed fields. Document length (sentences and tokens) and annotation counts (events and negative triggers) are right-skewed, with a majority of short to medium documents and a long tail of longer, more event-dense documents. These statistics describe the raw corpus before defining model-ready variables and support the choice of MAVEN for supervised event detection and the assumption that train and validation are comparable for evaluation.*



In [ ]:
def prevariable_stats(df):
    """
    Computes per-document statistics from a MAVEN event detection dataset dataframe.

    Parameters
    ----------
    df : pd.DataFrame
        Dataframe in MAVEN format: each row is a document with at least a `content` 
        (list of sentence dicts with "tokens"), and optionally columns 'events' and 
        'negative_triggers'.

    Returns
    -------
    pd.DataFrame
        Dataframe with one row per document containing the following columns:
            - n_sentences: Number of sentences in document
            - n_tokens: Number of tokens in document
            - vocab_size: Number of unique tokens (vocabulary size) in document
            - n_hapax: Number of tokens that occur only once (hapax legomena) in document
            - n_events: Number of events in document (if present)
            - n_negative_triggers: Number of negative triggers in document (if present)
    """
    # get the num of sentences, tokens, events, and negative triggers
    n_sentences = df["content"].map(len)
    n_tokens = df["content"].map(lambda c: sum(len(s["tokens"]) for s in c))
    n_events = df["events"].map(len) if "events" in df.columns else None
    n_neg = df["negative_triggers"].map(len) if "negative_triggers" in df.columns else None

    # For each document, extract tokens and compute vocabulary size and hapax count.
    def vocab_and_hapax(content_list):
        tokens = [t for sent in content_list for t in sent["tokens"]]
        fdist = FreqDist(tokens)
        return len(fdist), len(fdist.hapaxes())

    # Apply the function to each document in the dataframe
    vocab_size, n_hapax = zip(*df["content"].map(vocab_and_hapax))

    # Create a dataframe with the computed statistics
    out = pd.DataFrame({"n_sentences": n_sentences, "n_tokens": n_tokens, "vocab_size": vocab_size, "n_hapax": n_hapax})
    if n_events is not None:
        out["n_events"] = n_events
    if n_neg is not None:
        out["n_negative_triggers"] = n_neg
    return out

# Compute per-document stats
stats_train = prevariable_stats(df_train)
stats_val = prevariable_stats(df_val)

# One combined dataframe with split and doc_id
stats_train_labeled = stats_train.assign(split="train")
stats_val_labeled = stats_val.assign(split="valid")
stats_all = pd.concat([stats_train_labeled, stats_val_labeled], ignore_index=True)
stats_all.insert(0, "doc_id", np.concatenate([df_train["id"].values, df_val["id"].values]))

# Summary dataframe: train vs valid side by side (mean, std)
summary_side_by_side = pd.DataFrame({
    "train_mean": stats_train.mean(),
    "train_std": stats_train.std(),
    "valid_mean": stats_val.mean(),
    "valid_std": stats_val.std(),
})

# Print all the stats
print("\033[1;33m\nCombined per-document stats (first 15 rows):\033[0m"); display(stats_all.head(15))
print("\033[1;33m\nSummary (train vs valid):\033[0m"); display(summary_side_by_side)
print("\033[1;33m\nDescribe by split:\033[0m"); display(stats_all.groupby("split").describe())


# Define the columns to plot histograms
cols = ["n_sentences", "n_tokens", "n_events", "n_negative_triggers", "vocab_size", "n_hapax"]

# Plot for train
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, col in zip(axes.flat, cols):
    if col in stats_train.columns:
        sns.histplot(stats_train[col], kde=True, ax=ax, bins=30)
        ax.set_title(col)
    else:
        ax.axis('off')
plt.suptitle("Pre-variable distributions (train)", fontsize=12)
plt.tight_layout()
plt.show()

# Plot for validation
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, col in zip(axes.flat, cols):
    if col in stats_val.columns:
        sns.histplot(stats_val[col], kde=True, ax=ax, bins=30, color='orange')
        ax.set_title(col)
    else:
        ax.axis('off')
plt.suptitle("Pre-variable distributions (validation)", fontsize=12)
plt.tight_layout()
plt.show()

### Analysis 3: Univariate-level Overview

We look at one variable at a time: event type (and type_id) distribution across the corpus, trigger-word frequency, and the class balance between event triggers and negative triggers. This yields priors for event-type classification and for binary trigger detection, and shows the long-tail nature of the event ontology.

In [ ]:
def get_event_type_series(df):
    """Collect (type, type_id) for every event in df. Returns (type_series, type_id_series)."""
    all_types = []
    all_type_ids = []
    for _, row in df.iterrows():
        for ev in row["events"]:
            all_types.append(ev["type"])
            all_type_ids.append(ev["type_id"])
    return pd.Series(all_types), pd.Series(all_type_ids)

def get_trigger_balance(df):
    """Return (n_positive, n_negative, total) trigger spans."""
    n_positive = sum(len(ev.get("mention", [])) for _, row in df.iterrows() for ev in row["events"])
    n_negative = df["negative_triggers"].map(len).sum()
    total = n_positive + n_negative
    return n_positive, n_negative, total

def get_trigger_word_counts(df):
    """Return Series of trigger-word value counts."""
    words = []
    for _, row in df.iterrows():
        for ev in row["events"]:
            for m in ev.get("mention", []):
                w = m.get("trigger_word")
                if w is not None:
                    words.append(w)
    return pd.Series(words).value_counts()

def run_univariate_for_split(df, split_name, n_top=20, plot=True):
    """Run full univariate analysis for one split (train or valid)."""
    type_series, type_id_series = get_event_type_series(df)
    n_events = len(type_series)

    print(f"\033[1;33m\n{split_name.upper()}\033[0m")
    print(f"\033[1mEvent type (categorical) — value counts (top {n_top}):\033[0m")
    print(type_series.value_counts().head(n_top))
    print("\033[1m\nEvent type_id (numeric) — describe:\033[0m")
    print(type_id_series.describe())
    prior_top = type_series.value_counts().head(n_top) / n_events
    print(f"\033[1m\nPrior P(type) — top {n_top}:\033[0m")
    print(prior_top)

    n_pos, n_neg, total = get_trigger_balance(df)
    print(f"\033[1m\nTrigger vs negative:\033[0m \n - positive: {n_pos}, \n - negative: {n_neg}, \n - total: {total}")
    print(f"\033[1mPriors: \033[0m \n - P(positive) = {n_pos / total:.4f}, \n - P(negative) = {n_neg / total:.4f}")
    print("\033[1m\nTop 20 trigger words:\033[0m")
    print(get_trigger_word_counts(df).head(20))

    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        # Set color: train = blue, val = orange
        color = "blue" if split_name.lower() == "train" else "orange"
        type_series.value_counts().head(n_top).plot(kind="barh", ax=axes[0], color=color)
        axes[0].set_xlabel("Count")
        axes[0].set_ylabel("Event type")
        axes[0].set_title(f"Event type distribution ({split_name}, top {n_top})")
        type_id_series.value_counts().sort_index().plot(kind="bar", ax=axes[1], color=color)
        axes[1].set_xlabel("type_id")
        axes[1].set_ylabel("Count")
        axes[1].set_title(f"type_id distribution ({split_name})")
        plt.tight_layout()
        plt.show()

# Run for both splits
for name, data in [("train", df_train), ("valid", df_val)]:
    run_univariate_for_split(data, name, n_top=20, plot=True)

### Analysis 4: Baseline

We derive trivial baseline scores from the corpus priors (trigger vs negative and event-type distribution) so we have a lower bound any real model must beat. We then briefly reference the MAVEN benchmark (Wang et al., 2020): BERT+CRF reaches 67.8 F1 on MAVEN trigger classification; we use this model as our event extractor for the Tell Me Again pipeline and do not aim to improve MAVEN event detection itself.

*We use the MAVEN dataset for supervised event detection. From the training set, we computed trivial baselines: always predicting the negative class yields accuracy equal to the proportion of negative triggers (P(negative)), while always predicting the majority event type yields accuracy equal to the prior of the most frequent type. Any meaningful event detector must exceed these prior baselines. As the extractor, we rely on the BERT+CRF model reported by Wang et al. (2020), which achieves 67.8 F1 on MAVEN trigger classification. We do not aim to improve event detection on MAVEN but instead use it as an off-the-shelf component for our narrative similarity pipeline on the Tell Me Again dataset.*



In [ ]:
# Calculate priors for trigger vs negative trigger detection
n_pos, n_neg, total = get_trigger_balance(df_train)
p_neg = n_neg / total
p_pos = n_pos / total

# Calculate priors for event type classification
type_series, _ = get_event_type_series(df_train)
n_events = len(type_series)
majority_type = type_series.value_counts().index[0]
majority_count = type_series.value_counts().iloc[0]
p_majority_type = majority_count / n_events

print(f"\033[1;33m\nPrior baselines (train)\033[0m")
print(f"\033[1m  • Trigger vs negative:\033[0m \n - P(negative) = {p_neg:.4f} → 'always predict negative' accuracy = {p_neg:.4f}")
print(f"\033[1m  • Event type:\033[0m \n - majority type = '{majority_type}' (P = {p_majority_type:.4f}) → 'always predict majority' accuracy = {p_majority_type:.4f}")

# Print a summary table of the baseline accuracies
print("\033[1;33m\nBaseline accuracies from corpus priors (train set)\033[0m")
baseline_summary = pd.DataFrame({
    "Baseline": [
        "Always negative (trigger detection)",
        "Majority event type (type classification)",
    ],
    "Train accuracy": [p_neg, p_majority_type],
})
display(baseline_summary)

# 2) MAVEN paper benchmark (for methodology / related work)
print("\nMAVEN benchmark (Wang et al., 2020, Table 5 — trigger classification on MAVEN test):")
print("  • BERT+CRF: P=65.0, R=70.9, F1=67.8 (best in paper)")
print("  • We use BERT+CRF as the event extractor; our thesis focus is on Tell Me Again, not improving MAVEN ED.")

## Dataset 2: TellMeAgain!

### Data Loading

In [ ]:
# Define the core directories
PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname("."), ".."))
TELL_ME_AGAIN_ZIP_RAW = os.path.join(PROJECT_ROOT, "data", "raw", "TellMeAgain", "tell_me_again_v1.zip")

# Load the story-level corpus via zip file
storyDataset = StoryDataset(data_path=TELL_ME_AGAIN_ZIP_RAW)
# similarityDataset = SimilarityDataset()   # There is a recorded bug, so download for now (by leaving the data_path empty). TODO: change API code?

### Analysis 1: Corpus-level Overview

**In the cell below:**
- We iterate over each story in the TellMeAgain dataset.
- For each story, we collect various statistics, including the number of summaries in original and deduplicated form, presence of English or anonymized summaries, sentence count statistics, number of genres, and available languages.
- Each row in the resulting DataFrame represents a single story, containing all the collected statistics (e.g., Wikidata ID, number of summaries, languages, sentence counts, genres) for that story, allowing for easy analysis and display of the entire corpus.

**Observations:**
- Number of sentences with respect to languages differ: Although EN, DE, and FR have similar lengths/stds, IT and ES have significantly lower length.
- Singletons: Stories for which only one summary was collected in the original language, so there is only a single summary in the cluster.
- Deduplication: If the translated summaries of the same work has > 0.6 cosine similarity, the one with lower language-priority is filtered out at API runtime (`get_all_summaries_en()`)
- English summaries are not deduplicated as they are always the original summaries, not translations, so the 'n_deduped' will be 0 for them. 
- 423 stories did have multiple summaries, however the deduplication process removed all of these summaries, making them singletons. 


In [ ]:
# Build a statistics dataframe for the corpus where each row is a story and columns represent the corpus-level statistics
records = []
for story in storyDataset:

    # Raw cluster size (all original languages)
    numSummariesRaw = len(story.summaries_original)
    languagesRaw = list(story.summaries_original.keys())

    # Get the number of deduplicated summaries (i.e., summaries that become near-duplicates after translation are removed)
    ids_deduped, summaries_deduped = story.get_all_summaries_en()
    numSummariesDeduped = len(summaries_deduped)

    # Get the number of anonymized summaries
    numSummariesAnonymized = len(story.summaries_anonymized) if story.summaries_anonymized else 0

    # Get the sentence count statistics across all language versions of this story
    numSentences = list(story.num_sentences.values())
    avgSentences = sum(numSentences) / len(numSentences) if numSentences else 0
    maxSentences = max(numSentences) if numSentences else 0
    minSentences = min(numSentences) if numSentences else 0
    stdSentences = np.std(numSentences) if numSentences else 0

    # Append all of the statistics to the records
    records.append({
        "wikidata_id":         story.wikidata_id,                   # What is the wikidata id of the story?
        "n_summaries_raw":     numSummariesRaw,                     # How many summaries are there in the original language?
        "n_summaries_deduped": numSummariesDeduped,                 # How many summaries are there in the deduplicated language?
        "has_en":              "en" in story.summaries_original,    # Is English one of the original languages?
        "n_anon_langs":        numSummariesAnonymized,              # How many anonymized summaries are there?
        "has_anonymized":      numSummariesAnonymized > 0,          # Are there any anonymized summaries?
        "avg_sentences":       avgSentences,                        # What is the average number of sentences in the summaries?
        "max_sentences":       maxSentences,                        # What is the maximum number of sentences in the summaries?
        "min_sentences":       minSentences,                        # What is the minimum number of sentences in the summaries?
        "std_sentences":       stdSentences,                        # What is the standard deviation of the number of sentences in the summaries?
        "n_genres":            len(story.genres),                   # How many genres are there in the story?
        "langs":               languagesRaw,                        # What are the languages of the summaries?
    })

# Create a dataframe from the records dictionary
story_df = pd.DataFrame(records)

# Print the dataframe
story_df.head()

# Print the dataframe shape
print("\033[1;33m\nDataframe shape:\033[0m")
print(f"  {story_df.shape[0]:,} rows × {story_df.shape[1]} columns")

# Get the corpus dimensions
total_stories    = len(story_df)                                                # Number of unique works in the corpus.
total_raw        = story_df["n_summaries_raw"].sum()                            # Number of summaries in the raw corpus.
total_deduped    = story_df["n_summaries_deduped"].sum()                        # Number of summaries after deduplication.
singleton_count_before_dedup = (story_df["n_summaries_raw"] == 1).sum()         # unretrievable stories: Before deduplication, stories already have only one summary.
singleton_count_after_dedup  = (story_df["n_summaries_deduped"] == 1).sum()     # unretrievable stories: After deduplication, stories have only one summary left.

# Print the corpus dimensions
print("\033[1;33m\nCorpus dimensions:\033[0m")
print(f"  - Total stories (unique works):          {total_stories:>40,}")
print(f"  - Total summaries – raw (all languages): {total_raw:>40,}")
print(f"  - Total summaries – after deduplication: {total_deduped:>40,}")
print(f"  - Stories with an English summary:       {story_df['has_en'].sum():>40,}  ({story_df['has_en'].mean()*100:.1f}%)")
print(f"  - Stories with anonymized summaries:     {story_df['has_anonymized'].sum():>40,}  ({story_df['has_anonymized'].mean()*100:.1f}%)")
print(f"  - Singleton clusters (n_deduped == 1):   {singleton_count_after_dedup:>40,}  ({singleton_count_after_dedup/total_stories*100:.1f}%) \033[1m--> should be excluded from retrieval eval\033[0m")
print(f"  - Singleton clusters (number of summaries before deduplication == 1):   {singleton_count_before_dedup:>8,}  ({singleton_count_before_dedup/total_stories*100:.1f}%)  \033[1m--> should be excluded from retrieval eval\033[0m")

# Create a table of the summary count by language
lang_stats = storyDataset.get_lang_stats()
lang_df = (
    pd.DataFrame({
        "language":       list(lang_stats["languages"].keys()),
        "n_raw":          list(lang_stats["languages"].values()),
        "n_deduped":      [lang_stats["languages_direct_translations_removed"].get(k, 0)
                           for k in lang_stats["languages"].keys()],
    })
    .sort_values("n_raw", ascending=False)
    .reset_index(drop=True)
)

# Print the number of summaries per language
print("\033[1;33m\nNumber of summaries per language:\033[0m")
print("(Note: English summaries are not deduplicated as they are always the original summaries, not translations, so the 'n_deduped' will be 0)")
display(lang_df)

# Create a table of the summary length (sentences) per language version
print("\033[1;33m\nNumber of sentences per language version:\033[0m")
sent_records = []
for story in storyDataset:
    for lang, n in story.num_sentences.items():
        sent_records.append({"lang": lang, "n_sentences": n})
sent_df = pd.DataFrame(sent_records)
display(sent_df.groupby("lang")["n_sentences"].describe())

# Check missing values
no_anon = sum(1 for s in storyDataset if not s.summaries_anonymized)                                        # How many stories have no anonymized summaries?
no_genre = sum(1 for s in storyDataset if len(s.genres) == 0)                                               # How many stories have no genres?
no_en = sum(1 for s in storyDataset if "en" not in s.summaries_original)                                    # How many stories have no English original?
no_sim = sum(1 for s in storyDataset if len(s.similarities) == 0)                                           # How many stories have no similarities matrix?
no_summaries_original = sum(1 for s in storyDataset if not s.summaries_original)                            # How many stories have no original summaries?
no_summaries_translated = sum(1 for s in storyDataset if not s.summaries_translated)                        # How many stories have no translated summaries?
no_titles             = sum(1 for s in storyDataset if not s.titles)                                        # How many stories have no titles?
no_num_sentences      = sum(1 for s in storyDataset if not s.num_sentences)                                 # How many stories have no sentence counts?
no_sentences          = sum(1 for s in storyDataset if not s.sentences)                                     # How many stories have no sentences?
empty_sim_labels      = sum(1 for s in storyDataset if not (getattr(s, "similarities_labels", None) or [])) # How many stories have no similarity labels?
no_description        = sum(1 for s in storyDataset if not (s.description or "").strip())                   # How many stories have no descriptions?
def _title_string(s):
    """Return a usable title string for a story, using 'title' or entries in 'titles'."""
    t = getattr(s, "title", None)
    if isinstance(t, str) and t.strip():
        return t.strip()
    if isinstance(t, dict) and t.get("value"):
        return str(t["value"]).strip()
    titles = getattr(s, "titles", None) or {}
    for v in (titles or {}).values():
        if v and str(v).strip():
            return str(v).strip()
    return ""
no_usable_title = sum(1 for s in storyDataset if not _title_string(s))


# Print the missing values
print("\033[1;33m\nMissing values:\033[0m")
print(f"  - Stories with no anonymized summaries: {no_anon}")
print(f"  - Stories with no genres: {no_genre}")
print(f"  - Stories with no English original (but can have multiple EN summaries): {no_en}")
print(f"  - Stories with no similarities matrix: {no_sim}")
print(f"  - Stories with empty summaries_original:   {no_summaries_original}")
print(f"  - Stories with empty summaries_translated: {no_summaries_translated}")
print(f"  - Stories with empty titles dict:          {no_titles}")
print(f"  - Stories with empty num_sentences:       {no_num_sentences}")
print(f"  - Stories with empty sentences:            {no_sentences}")
print(f"  - Stories with empty similarities_labels: {empty_sim_labels}")
print(f"  - Stories with blank description:        {no_description}")
print(f"  - Stories with blank title:               {no_usable_title}")


# Plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Raw cluster size distribution
cs_raw = story_df["n_summaries_raw"].value_counts().sort_index()
cs_raw.plot(kind="bar", ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Cluster size – raw (all languages)")
axes[0, 0].set_xlabel("# summaries per story"); axes[0, 0].set_ylabel("# stories")

# Deduplicated cluster size distribution (what retrieval eval sees)
cs_ded = story_df["n_summaries_deduped"].value_counts().sort_index()
cs_ded.plot(kind="bar", ax=axes[0, 1], color="darkorange")
axes[0, 1].set_title("Cluster size – after deduplication")
axes[0, 1].set_xlabel("# summaries per story (deduped)"); axes[0, 1].set_ylabel("# stories")

# Language distribution
lang_df.plot(x="language", y=["n_raw", "n_deduped"], kind="bar", ax=axes[0, 2])
axes[0, 2].set_title("Summary count by language")
axes[0, 2].set_xlabel("Language"); axes[0, 2].set_ylabel("# summaries")
axes[0, 2].legend(["raw", "after dedup"])

# Sentence-count distribution per summary (EN only, then all)
sns.histplot(sent_df[sent_df["lang"] == "en"]["n_sentences"],
             kde=True, bins=50, ax=axes[1, 0], color="steelblue", 
             label="en", stat="density")
sns.histplot(sent_df[sent_df["lang"] != "en"]["n_sentences"],
             kde=True, bins=50, ax=axes[1, 0], color="darkorange", 
             alpha=0.6, label="non-en", stat="density")
axes[1, 0].set_title("Sentence count per summary (EN vs non-EN)")
axes[1, 0].set_xlabel("# sentences"); axes[1, 0].legend()

# Anonymized language coverage
anon_dist = story_df["n_anon_langs"].value_counts().sort_index()
anon_dist.plot(kind="bar", ax=axes[1, 1], color="mediumseagreen")
axes[1, 1].set_title("# anonymized versions per story")
axes[1, 1].set_xlabel("# anonymized languages"); axes[1, 1].set_ylabel("# stories")

# Genre count per story
story_df["n_genres"].value_counts().sort_index().plot(kind="bar", ax=axes[1, 2], color="mediumpurple")
axes[1, 2].set_title("Wikidata genre count per story")
axes[1, 2].set_xlabel("# genres"); axes[1, 2].set_ylabel("# stories")

plt.suptitle("Analysis 1: Corpus-Level Overview – TellMeAgain!", fontsize=14)
plt.tight_layout()
plt.show()

# One full example Story
ex = list(storyDataset)[0]
ids_ex, sums_ex = ex.get_all_summaries_en()
print(f"\033[1;33m\nExample Story:\033[0m \n{ex}")
print(f"  wikidata_id:   {ex.wikidata_id}")
print(f"  title:         {ex.title}")
print(f"  description:   {ex.description}")
print(f"  Languages present: {list(ex.summaries_original.keys())}")
print(f"  num_sentences:     {ex.num_sentences}")
print(f"  genres (Q-IDs):    {ex.genres}")
print(f"  anonymized langs:  {list(ex.summaries_anonymized.keys()) if ex.summaries_anonymized else 'None'}")
print(f"  get_all_summaries_en → {len(ids_ex)} summaries after dedup")
print(f"\n  EN summary (first 400 chars):\n  {ex.summaries_original.get('en', '[no EN]')[:400]}...")

# Print all example summaries
print("\n\033[1;33mExample summaries:\033[0m")
for i, summary in enumerate(sums_ex):
    print(f"\n  Summary {i+1}:")
    print(f"    {summary}")

### Analysis 2: Inspecting Stories with No Genres
We are exploring stories that are missing genre information and demonstrating how to identify them in the dataset in the next cell.

**Results:** 
- If a stratified sampling will be done, these stories can be removed from the dataset. 
- Also, if the train/test/val split is not representative of the full dataset with respect to Genre, these can be removed as well.

In [ ]:
# Filter stories with no genres
stories_no_genre = [s for s in storyDataset if len(s.genres) == 0]
print(f"\nThere are {len(stories_no_genre)} stories with NO genres.")

if stories_no_genre:
    example_no_genre = stories_no_genre[0]
    print("\033[1;33m\nExample story with no genres:\033[0m")
    fields = [
        "wikidata_id", "title", "titles", "description",
        "summaries_original", "summaries_translated", "summaries_anonymized",
        "num_sentences", "sentences", "genres", "similarities_labels"
    ]
    for field in fields:
        print(f"\n--- {field} ---")
        value = getattr(example_no_genre, field, None)
        if isinstance(value, (dict, list)):
            import pprint
            pprint.pprint(value)
        else:
            print(value)
    
    print("\n--- similarities shape ---")
    similarities = getattr(example_no_genre, "similarities", None)
    if hasattr(similarities, "shape"):
        print(similarities.shape)
    else:
        print(type(similarities))
else:
    print("No stories without genres found.")

### Analysis 3: Inspecting Stories with No English original but have >=2 EN summaries
 
This is the case where stories do not have an original English summary, but there are at least two English summaries available from translations or other sources. In the next cell, we identify such stories and explain what should be kept or removed.
 
**Results:** 
- These stories can be kept for retrieval task.
- Delete the ones that have no EN original, but also don't have >=2 EN summaries. 

In [ ]:
# Inspect stories that truly have no English original summary (not just based on cluster counts)
no_en_stories = [s for s in storyDataset if "en" not in s.summaries_original]
print(f"\nThere are {len(no_en_stories)} stories with NO English original summary.")

# Find how many of those have >=2 EN summaries (across all sources, including deduped translations)
no_en_multiple_en = [
    s for s in no_en_stories if len(s.get_all_summaries_en()[0]) >= 2
]
print(f"Out of these, {len(no_en_multiple_en)} stories have >=2 EN summaries (after deduplication, across all sources).")

# Optionally: show an example and explore their EN (translated) deduped summaries, if any
example_no_en = None
for s in no_en_multiple_en:
    example_no_en = s
    break

if example_no_en is not None:
    print("\033[1;33m\nExample story with no native EN but multiple EN summaries (from translations only):\033[0m")
    fields = [
        "wikidata_id", "title", "titles", "description",
        "summaries_original", "summaries_translated", "summaries_anonymized",
        "num_sentences", "sentences", "genres", "similarities_labels"
    ]
    for field in fields:
        print(f"\n--- {field} ---")
        value = getattr(example_no_en, field, None)
        if isinstance(value, (dict, list)):
            pprint.pprint(value)
        else:
            print(value)

    print("\n--- similarities shape ---")
    similarities = getattr(example_no_en, "similarities", None)
    if hasattr(similarities, "shape"):
        print(similarities.shape)
    else:
        print(type(similarities))

    ids_en, summaries_en = example_no_en.get_all_summaries_en()
    print(f"\n--- get_all_summaries_en() ---")
    print(f"  {len(ids_en)} EN summaries (ids: {ids_en})")
else:
    print("No example with multiple EN summaries found among stories lacking a native EN summary.")

### Analysis 4: Inspecting stories with no translated summaries
 
Here we inspect stories that lack an original English summary, but have multiple deduplicated English summaries originating from translations or other sources. Below, we identify such stories, show an example, and examine the relevant fields.
 
**Results:**
- Remove this story from the final dataset.

In [ ]:
# Inspect how many stories have no translated summaries
no_summaries_translated = sum(1 for s in storyDataset if not s.summaries_translated)
print(f"\nThere are {no_summaries_translated} stories with NO translated summaries.")

if no_summaries_translated > 0:
    # Show an example story with no translated summaries
    example_no_translated = next(s for s in storyDataset if not s.summaries_translated)
    print("\033[1;33m\nExample story with no translated summaries:\033[0m")
    fields = [
        "wikidata_id", "title", "titles", "description",
        "summaries_original", "summaries_translated", "summaries_anonymized",
        "num_sentences", "sentences", "genres", "similarities_labels"
    ]
    for field in fields:
        print(f"\n--- {field} ---")
        value = getattr(example_no_translated, field, None)
        if isinstance(value, (dict, list)):
            import pprint
            pprint.pprint(value)
        else:
            print(value)
    
    print("\n--- similarities shape ---")
    similarities = getattr(example_no_translated, "similarities", None)
    if hasattr(similarities, "shape"):
        print(similarities.shape)
    else:
        print(type(similarities))
else:
    print("No stories without translated summaries found.")

### Analysis 5: Inspecting stories with no blank description
 
We inspect stories with a blank or missing description, count how many there are, and show an example with relevant fields.
 
**Results:** 
- Since description is not a key variable in our pipline, these stories can be kept. 
- However, if stratified sampling will be done, and description is part of it, then remove those

In [ ]:
# Inspect how many stories have a blank (missing or just whitespace) description
no_description = sum(1 for s in storyDataset if not (s.description or "").strip())
print(f"\nThere are {no_description} stories with BLANK description.")

if no_description > 0:
    # Show an example story with a blank description
    example_blank_desc = next(s for s in storyDataset if not (s.description or "").strip())
    print("\033[1;33m\nExample story with blank description:\033[0m")
    fields = [
        "wikidata_id", "title", "titles", "description",
        "summaries_original", "summaries_translated", "summaries_anonymized",
        "num_sentences", "sentences", "genres", "similarities_labels"
    ]
    for field in fields:
        print(f"\n--- {field} ---")
        value = getattr(example_blank_desc, field, None)
        if isinstance(value, (dict, list)):
            import pprint
            pprint.pprint(value)
        else:
            print(value)
    
    print("\n--- similarities shape ---")
    similarities = getattr(example_blank_desc, "similarities", None)
    if hasattr(similarities, "shape"):
        print(similarities.shape)
    else:
        print(type(similarities))
else:
    print("No stories with blank description found.")

### Analysis 6: Pipeline Eligibility (EN-only)

Since MAVEN is English-only, our pipeline operates exclusively on English summaries — either native English Wikipedia summaries or non-English summaries machine-translated to English via `get_all_summaries_en()`. This analysis characterises the EN-projected view of the corpus: how many works have multiple EN summaries (i.e., are retrievable), how the EN cluster sizes are distributed, which source-language combinations produce the EN pool, and how summary lengths compare between native and translated EN.

**Results:**
- result 1
- result 2

### Analysis 7: Token Size - Context Window Analysis 

This analysis examines the average token size of eligible summaries in our dataset for Llama-3-8B-Instruct. Understanding token distribution helps assess how well our data fits within the model's context window.

**Results:**
- result 1
- result 2

### Analysis 8: Split Representation Analysis

The main purpose here is to ensure that downstream experiments in the English pipeline are valid and that evaluation and training are carried out on appropriate, representative subsets.

**This section will:**
- Examine which split(s) were used in the original TellMeAgain! dataset and their characteristics.
- Analyze whether the official split is representative for our *pipeline-eligible* (EN-only) subset — i.e., those summaries that would be eligible for our English-only MAVEN pipeline.
- Assess what proportion of the split is actually pipeline-eligible and where non-eligibility arises.
- Explore alternative splitting strategies and compare their characteristics: for example, how many works and summaries are available in each split if we restrict to pipeline-eligible cases, and whether new stratified or random splits might offer advantages.

**Results:**
- result 1
- result 2

### Analysis 9: Baseline from Original Distribution

This section establishes reference baselines for our similarity task *without* using any trained or pretrained models. Instead, we focus on baselines derived only from the empirical distribution of similarity scores and class labels in our dataset:
- Compute the distribution of pairwise similarity scores (or class labels, if pairs are already labeled).
- Determine the class balance: the proportion of "positive" (similar) versus "negative" (dissimilar) pairs under a chosen similarity threshold or annotation definition.
- Calculate the accuracy of a simple majority-class predictor, which always guesses the most common class in the data.
- Report the resulting expected baseline accuracy. Any system must exceed this value to demonstrate skill.